# 從單一 agent 到 LangGraph workflow：寫你自己的多 agent 編排器

在前一份 notebook（`adding_tools_to_agents.ipynb`）我們做了一個 `react_agent`：給它幾個工具，它就會自己思考、自己呼叫工具來回答問題。這已經很厲害，但它有兩個我們在實務上會踩到的限制：

1. **它對所有 query 一視同仁**：你問「你好」，它也會跑一遍完整的 LLM 工具判斷流程，浪費 token；你問「幫我寫一首詩」，它仍可能勉強去呼叫不相關的工具。
2. **多功能會把單一 agent 撐爆**：當你的助手要做的事情很多（搜尋、撰寫、計算、客服…），把所有工具塞給同一個 react_agent，反而會讓它的判斷品質下降——上下文裡的工具描述變多、LLM 在「這次該選哪個」上會越來越不準。

實務上更好的做法是「**分工**」：先用一個輕量的**分類器**判斷意圖，再把工作交給該領域的**專業 sub-agent**。這就是業界俗稱的 **multi-agent routing / orchestration** 模式——也是大多數 production 級 agentic 系統的骨架。

但要怎麼把這種「先分類、再分發」的流程乾淨地寫出來？純 Python `if/else` 雖然能跑，可一旦節點變多、分支變複雜、又想加 retry / checkpoint，就會難以維護。所以本份 notebook 要介紹的工具叫 [**LangGraph**](https://langchain-ai.github.io/langgraph/)——一個用「**狀態圖（state graph）**」描述 workflow 的小框架：你把每個步驟畫成一個節點、節點之間用邊串起來、條件邊處理分岔；寫出來的工作流程**看起來就像流程圖**，最後一行 `compile()` 就把它變成一個可以執行的 async function。

我們的目標：

- 寫兩個小工具：`weather_lookup`（查 wttr.in 真實天氣）與 `nvidia_blog_search`（查 NVIDIA Developer Blog 最新文章）
- 把它們分別包成兩個 `tool_calling_agent` 子代理人
- **用 LangGraph 親手寫一個 NAT workflow function**：把「意圖分類 → 路由到對應 sub-agent → 回覆」整段流程畫成狀態圖的節點與邊
- 把整張圖登記成一個全新的 `_type: simple_router_agent`，跟內建 agent 等價地被 NeMo Agent Toolkit 認得

完成後你會具備一個對寫進階 agent 系統很關鍵的能力：**用 LangGraph 把任意多個 sub-agent 組裝成自己的 workflow**。這是從「會寫單一 react_agent」邁向「能設計多 agent 編排系統」之間最重要的一塊拼圖。


# 目錄

- [0.0) 環境準備](#setup)
  - [0.1) 系統需求](#prereqs)
  - [0.2) API 金鑰與 LangGraph 套件](#api-keys)
- [1.0) 概念：workflow 是一張「狀態圖」](#concept)
  - [1.1) NAT 提供的兩種寫 workflow 的方式](#two-ways)
  - [1.2) 第一次認識 LangGraph](#langgraph-intro)
- [2.0) 場景：給多功能小幫手加上意圖路由](#scenario)
  - [2.1) 為什麼純 `react_agent` 不夠用](#why-router)
  - [2.2) 我們要建的狀態圖](#what-to-build)
- [3.0) 建立 workflow 專案 + 兩個工具](#tools)
  - [3.1) 用 `nat workflow create` 建立骨架](#scaffold)
  - [3.2) 工具 1：`weather_lookup`](#tool-weather)
  - [3.3) 工具 2：`nvidia_blog_search`](#tool-blog)
- [4.0) 用 LangGraph 寫 `simple_router_agent`](#router)
  - [4.1) LangGraph 三件套：State、Node、Edge](#three-pieces)
  - [4.2) 設計 `RouterState`（狀態）](#design-state)
  - [4.3) 設計 5 個節點](#design-nodes)
  - [4.4) 設計條件邊：`route_by_intent`](#design-edges)
  - [4.5) 把所有東西包成 NAT workflow function](#nat-wrap)
  - [4.6) 註冊、安裝、確認](#router-install)
- [5.0) 把所有東西串成一份 YAML](#yaml)
  - [5.1) Before：所有工具塞給單一 agent](#yaml-before)
  - [5.2) After：兩個 sub-agent + 一個 LangGraph router](#yaml-after)
- [6.0) 跑跑看](#run)
  - [6.1) Meta query：`classify → meta → END`](#run-meta)
  - [6.2) Weather query：`classify → weather → END`](#run-weather)
  - [6.3) Blog query：`classify → blog → END`](#run-blog)
  - [6.4) Unknown query：`classify → unknown → END`](#run-unknown)
- [7.0) 總結與下一步](#bridge-to-aiq)

<span style="color:rgb(0, 31, 153); font-style: italic;">小提醒：如果你是在 Google Colab 上閱讀，建議切到左邊的「目錄」分頁來瀏覽，比較好導覽。</span>


<a id="setup"></a>
# 0.0) 環境準備

這份 notebook 假設你**已經跑過** `adding_tools_to_agents.ipynb`。也就是：你在 `examples/notebooks/zh-TW/` 底下已經有一個可以用的 `.venv`，並且 `nat` CLI 跑得起來。


<a id="prereqs"></a>
## 0.1) 系統需求

- 一個能跑 NAT 的 Python 環境（接續 `adding_tools_to_agents.ipynb` 的 `.venv` 即可）
- `nat` CLI 已安裝（前一份 notebook 已經裝過 NeMo Agent Toolkit）
- NVIDIA Build API 金鑰
- [`langgraph`](https://github.com/langchain-ai/langgraph) Python 套件（下面會一起裝）


<a id="api-keys"></a>
## 0.2) API 金鑰與 LangGraph 套件

跟前一份 notebook 一樣，我們會用 NVIDIA Build 的雲端 LLM。如果你已經把 `NVIDIA_API_KEY` 設好就可以直接往下跑。


In [1]:
import getpass
import os

if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("請輸入你的 NVIDIA Build API 金鑰：")

print("NVIDIA_API_KEY 已設定：", os.environ.get("NVIDIA_API_KEY", "")[:6] + "...")

請輸入你的 NVIDIA Build API 金鑰： ········


NVIDIA_API_KEY 已設定： nvapi-...


LangGraph 不是 NAT 的硬性相依，所以我們先把它裝起來。如果你的環境已經裝過（例如裝過 AI-Q），這個 cell 也會直接 no-op。


In [2]:
!pip install langgraph --quiet

<a id="concept"></a>
# 1.0) 概念：workflow 是一張「狀態圖」

在 NAT 的世界裡，`workflow` 跟 `function` 其實**是同一種東西**——它們都是用 `@register_function` 註冊出來的、有一個 `_type` 名字的 Python 函式。差別只是：

- 寫在 YAML `functions:` 底下 → 我們叫它 **tool**（或 sub-agent）
- 寫在 YAML `workflow:` 底下 → 我們叫它 **workflow**（整個系統的入口）

而你前一份 notebook 用的 `react_agent`，**就是一個 NAT 內建的 workflow function**。它的 `_type` 是 `react_agent`，跟你自己寫的工具 `_type: get_total_product_sales_data` 在 NAT 眼裡完全是同一個概念。

**換句話說**：

> 如果你會寫 NAT 工具（你已經會了），你就會寫 NAT workflow。


<a id="two-ways"></a>
## 1.1) NAT 提供的兩種寫 workflow 的方式

| | 方式 A：直接用內建 agent | 方式 B：自己寫一個 NAT function 當 workflow |
| --- | --- | --- |
| YAML 怎麼寫？ | `workflow: { _type: react_agent, tool_names: [...] }` | `workflow: { _type: my_workflow_name, ...任意參數 }` |
| Python 端要做什麼？ | **什麼都不用**（react_agent 已經寫好了） | **自己寫一個** `@register_function`，內部用 `builder.get_function("...")` 取出別人的 sub-agent / tool 來組合 |
| 適用情境 | 「一個 agent + 一堆工具」 | 多 agent 編排、流程控制、客製化路由、HITL |

前一份 notebook **全程都在用方式 A**。AI-Q 則**全程都在用方式 B**——它的 `_type: chat_deepresearcher_agent` 就是一個 NVIDIA 自己寫的 workflow function。

當你選擇方式 B、要寫一個有「分支、迴圈、需要保存中介狀態」的 workflow 時，你需要一個工具來**描述這個流程**。手寫 if/else 可以，但**很快會變成一團義大利麵**。這就是 LangGraph 登場的時機。


<a id="langgraph-intro"></a>
## 1.2) 第一次認識 LangGraph

[LangGraph](https://langchain-ai.github.io/langgraph/) 是 LangChain 團隊推出的**狀態機**框架。它的核心想法是：

> **把你的 workflow 畫成一張圖：節點（node）是會跑事情的函式，邊（edge）是節點之間的轉移規則。整張圖共用一份「狀態」（state），每個節點看一眼狀態、做事、然後更新狀態，再交給下一個節點。**

聽起來很抽象？看下面這張對照表就會懂：

| 你直覺會這樣寫… | 用 LangGraph 寫… |
| --- | --- |
| `if intent == "weather": ... elif intent == "date": ...` | 一個 `classify` node + 一個**條件邊**（conditional edge）依 `state.intent` 決定下一站 |
| `state = {"query": q}; state["intent"] = classify(q); state["answer"] = run_agent(...)` | 一個 `RouterState` Pydantic 物件，自動在節點之間流動 |
| 「Shallow 失敗時 escalate 到 Deep」 | 從 shallow node 拉一條 conditional edge 到 deep node |

**為什麼要用 LangGraph 而不是 if/else？**

| 用 if/else | 用 LangGraph |
| --- | --- |
| 程式碼長大就會難維護 | 節點與邊**分開定義**，可視覺化、可重構 |
| 自己管狀態很容易出 bug | 狀態用 Pydantic 自動驗證、節點之間自動 merge |
| 中斷恢復、checkpoint 要自己寫 | LangGraph 內建 [checkpointer](https://langchain-ai.github.io/langgraph/concepts/persistence/) |
| HITL（暫停問人）要自己接 | LangGraph 內建 [interrupt 機制](https://langchain-ai.github.io/langgraph/concepts/human_in_the_loop/) |
| 多 agent 編排會變成大型義大利麵 | 圖一目了然，加減節點/邊就能擴充 |

**這也是為什麼 AI-Q 選 LangGraph**。我們這份 notebook 會親手用 LangGraph 寫一個迷你版的 router，等你下一份看到 AI-Q 的 `_build_graph()` 時，就會發現結構**一模一樣**。


<a id="scenario"></a>
# 2.0) 場景：給多功能小幫手加上意圖路由


<a id="why-router"></a>
## 2.1) 為什麼純 `react_agent` 不夠用

想像我們做了一個小幫手，裡面塞了兩個工具：`weather_lookup`（查天氣）跟 `nvidia_blog_search`（查 NVIDIA blog）。如果直接用 react_agent 包起來，會發生什麼事？

| 使用者問 | react_agent 做的事 | 問題 |
| --- | --- | --- |
| 「你好」 | 跑一次 LLM 推論，判斷不用工具，回覆 | 浪費一次 LLM 呼叫 |
| 「台北今天天氣？」 | LLM 推論 → 呼叫 `weather_lookup` → 回覆 | ✅ 正常 |
| 「幫我寫一首詩」 | LLM 推論可能誤判 → 呼叫某個工具失敗 → retry → 最終回覆 | 浪費 token，可能還回答錯誤 |

當工具越多、工具的 schema 越複雜，這種「亂用工具」與「無謂呼叫」的問題會越來越嚴重。**這也是為什麼 AI-Q 一進來就先用一個 `intent_classifier` 把 query 分類好**，再決定要走哪一條路——而做這個「決定要走哪一條路」的，正是 LangGraph 的條件邊。


<a id="what-to-build"></a>
## 2.2) 我們要建的狀態圖

我們要用 LangGraph 蓋出下面這張圖：

```mermaid
graph LR
    START([START]) --> C[classify_node<br/>LLM 意圖分類]
    C -.weather.-> WN[weather_node<br/>呼叫 weather_agent]
    C -.blog.-> BN[blog_node<br/>呼叫 blog_agent]
    C -.meta.-> MN[meta_node<br/>固定回覆]
    C -.unknown.-> UN[unknown_node<br/>禮貌拒絕]
    WN --> END([END])
    BN --> END
    MN --> END
    UN --> END

    style C fill:#fff3e0
    style WN fill:#f3e5f5
    style BN fill:#e8eaf6
    style MN fill:#fce4ec
    style UN fill:#fce4ec
```

**讀法**：

- 圓角矩形是 LangGraph **節點**（node）
- 實線箭頭是**普通邊**（unconditional edge）：跑完一定走這條
- 虛線箭頭是**條件邊**（conditional edge）：跑完之後由一個函式決定走哪條，箭頭上的字（`weather` / `blog` / ...）就是那個函式可能回傳的標籤

**對應到 NAT 的層次**：

- `weather_lookup`、`nvidia_blog_search` → 兩個 **tool**
- `weather_agent`、`blog_agent` → 兩個 **NAT 內建 agent**（這份 notebook 用 `tool_calling_agent`，每個都是「方式 A」風格的 sub-workflow）
- `simple_router_agent` → 一個用 LangGraph 寫的 **workflow function**（「方式 B」）

如果你拿這張圖跟 AI-Q `chat_deepresearcher_agent` 的流程圖對比，你會發現它們**結構幾乎一模一樣**——AI-Q 只是節點變多、邊變複雜（例如淺層 escalate 到深層）。


<a id="tools"></a>
# 3.0) 建立 workflow 專案 + 兩個工具


<a id="scaffold"></a>
## 3.1) 用 `nat workflow create` 建立骨架

跟前一份 notebook 同樣的招式：先用 NAT CLI 幫我們把專案目錄骨架蓋出來。


In [3]:
!nat workflow create router_agent_nb

Installing workflow 'router_agent_nb'...
Workflow 'router_agent_nb' installed successfully.
Workflow 'router_agent_nb' created successfully in '/localhome/local-clchiu/NeMo-Agent-Toolkit/examples/notebooks/zh-TW/router_agent_nb'.


執行後你應該會看到 `router_agent_nb/` 這個資料夾，裡面有 `pyproject.toml`、`src/router_agent_nb/`、`configs/` 等子目錄。接下來我們會把工具與 router 的程式碼用 `%%writefile` 寫進這個專案裡。


<a id="tool-weather"></a>
## 3.2) 工具 1：`weather_lookup`

一個查天氣的工具，這次我們**真的去查網路上的即時天氣**，使用免費、不需要 API 金鑰的 [wttr.in](https://wttr.in) 服務。

這也順便展示了一個重要觀念：**NAT tool 不限於本地計算，也可以呼叫外部 API**。前一份 notebook 的 sales tool 是讀本地 CSV，這份的 `weather_lookup` 是打 HTTP；AI-Q 的 `web_search` 是呼叫 Tavily / SerpAPI；RAG 工具則是查向量資料庫——在 NAT 眼裡這些**全部都是同一種 function**，差別只是內部實作。

另外順便示範：**NAT config 可以拿來調工具的行為**。我們把 `units`（公制或英制）做成 YAML 可配置的欄位，不用改 Python 就能切換攝氏 / 華氏。


In [4]:
%%writefile router_agent_nb/src/router_agent_nb/weather_lookup_tool.py
"""Weather lookup tool that queries wttr.in for real-time data."""
import asyncio
import logging
import urllib.parse
import urllib.request

from pydantic import Field

from nat.builder.builder import Builder
from nat.builder.function_info import FunctionInfo
from nat.cli.register_workflow import register_function
from nat.data_models.function import FunctionBaseConfig

logger = logging.getLogger(__name__)


# 註：docstring 與工具參數說明會傳給 LLM 作為工具描述，故保留英文
class WeatherLookupConfig(FunctionBaseConfig, name="weather_lookup"):
    """Look up the current weather for a city from wttr.in (no API key needed)."""
    units: str = Field(
        default="m",
        description="Unit system: 'm' for metric (°C, km/h) or 'u' for US imperial (°F, mph).",
    )
    timeout_seconds: float = Field(
        default=10.0,
        description="HTTP request timeout in seconds.",
    )


# wttr.in 預設會看 User-Agent 決定回傳 HTML 還是純文字；給 curl-like UA 就能拿到純文字。
_USER_AGENT = "curl/7.88.0"
# Custom format: location: condition temp, wind <wind>, humidity <humidity>
_WTTR_FORMAT = "%l:+%C+%t,+wind+%w,+humidity+%h"


def _fetch_weather_sync(url: str, timeout: float) -> str:
    req = urllib.request.Request(url, headers={"User-Agent": _USER_AGENT})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read().decode("utf-8", errors="replace")


@register_function(config_type=WeatherLookupConfig)
async def weather_lookup_function(config: WeatherLookupConfig, _builder: Builder):
    """Real-time weather lookup powered by wttr.in."""
    units_flag = "u" if config.units.lower().startswith("u") else "m"

    async def _lookup(city: str) -> str:
        """Look up current weather for a city using the free wttr.in service.

        Args:
            city: City name in English (e.g. "Taipei", "Tokyo", "New York").

        Returns:
            A short human-readable weather summary, or an error message.
        """
        if not city or not city.strip():
            return "Please provide a city name."

        safe_city = urllib.parse.quote(city.strip())
        url = f"https://wttr.in/{safe_city}?format={_WTTR_FORMAT}&{units_flag}"
        logger.info("[weather_lookup] GET %s", url)

        try:
            text = await asyncio.to_thread(_fetch_weather_sync, url, config.timeout_seconds)
        except Exception as exc:
            return f"Failed to look up weather for '{city}': {exc}"

        text = text.strip()
        if not text or text.lower().startswith("unknown location"):
            return f"No weather data found for '{city}'."
        return text

    yield FunctionInfo.from_fn(_lookup, description=_lookup.__doc__)


Writing router_agent_nb/src/router_agent_nb/weather_lookup_tool.py


<a id="tool-blog"></a>
## 3.3) 工具 2：`nvidia_blog_search`

第二個工具會去 [NVIDIA Developer Blog](https://developer.nvidia.com/blog) 抓最新文章，可以選擇用關鍵字篩選。我們呼叫的是它官方的 [WordPress JSON API](https://developer.nvidia.com/blog/wp-json/wp/v2/posts)——這個 endpoint 不需要 API 金鑰，而且回的是乾淨的 JSON（比 RSS feed 容易解析得多）。

這個工具也順便示範一個給 LLM 寫工具時很重要的觀念：**LLM 看到的工具輸出，最好是「結構化但人類可讀」的純文字**，而不是原始 JSON 或 HTML。WordPress API 的 excerpt 欄位塞了一堆 `<p>`、`&hellip;`、HTML entity，直接丟給 LLM 會嚴重浪費 context window；所以我們在這個工具裡會做兩件事：

1. 用 `_fields` 參數**只抓我們要的欄位**（date、link、title、excerpt），減少網路傳輸與下游噪音
2. 在 Python 端**清掉 HTML tag、unescape entity、收斂空白**，最後輸出一份編號清單給 LLM


In [5]:
%%writefile router_agent_nb/src/router_agent_nb/nvidia_blog_search_tool.py
"""Tool that fetches the latest posts from the NVIDIA Technical Blog."""
import asyncio
import html
import json
import logging
import re
import urllib.parse
import urllib.request

from pydantic import Field

from nat.builder.builder import Builder
from nat.builder.function_info import FunctionInfo
from nat.cli.register_workflow import register_function
from nat.data_models.function import FunctionBaseConfig

logger = logging.getLogger(__name__)

_API_URL = "https://developer.nvidia.com/blog/wp-json/wp/v2/posts"
_USER_AGENT = "Mozilla/5.0 (compatible; nvidia-blog-search-tool/0.1)"


# 註：docstring 與工具參數說明會傳給 LLM 作為工具描述，故保留英文
class NvidiaBlogSearchConfig(FunctionBaseConfig, name="nvidia_blog_search"):
    """Search the NVIDIA Developer Blog (no API key needed)."""
    max_results: int = Field(
        default=5,
        ge=1,
        le=10,
        description="Maximum number of blog posts to return (1-10).",
    )
    excerpt_max_chars: int = Field(
        default=240,
        description="Trim each excerpt to at most this many characters.",
    )
    timeout_seconds: float = Field(
        default=10.0,
        description="HTTP request timeout in seconds.",
    )


def _fetch_sync(url: str, timeout: float) -> str:
    req = urllib.request.Request(url, headers={"User-Agent": _USER_AGENT})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read().decode("utf-8", errors="replace")


_HTML_TAG_RE = re.compile(r"<[^>]+>")
_WHITESPACE_RE = re.compile(r"\s+")


def _clean_text(raw: str) -> str:
    """Strip HTML tags, unescape entities, collapse whitespace, drop trailing 'Continued'."""
    text = _HTML_TAG_RE.sub("", raw)
    text = html.unescape(text)
    text = _WHITESPACE_RE.sub(" ", text).strip()
    for tail in ("Continued", "…"):
        if text.endswith(tail):
            text = text[: -len(tail)].rstrip(" .")
    return text


@register_function(config_type=NvidiaBlogSearchConfig)
async def nvidia_blog_search_function(config: NvidiaBlogSearchConfig, _builder: Builder):
    """Fetch latest posts from the NVIDIA Developer Blog, optionally filtered by keywords."""

    async def _search(query: str = "") -> str:
        """Search the NVIDIA Developer Blog (developer.nvidia.com/blog).

        Args:
            query: Optional keywords to filter posts (e.g. "Blackwell", "RAG", "agentic AI").
                   If empty, returns the most recent posts.

        Returns:
            A numbered list of blog posts with date, title, URL, and short excerpt.
        """
        params = {
            "per_page": str(config.max_results),
            "_fields": "date,link,title,excerpt",
        }
        if query and query.strip():
            params["search"] = query.strip()
        url = f"{_API_URL}?{urllib.parse.urlencode(params)}"
        logger.info("[nvidia_blog_search] GET %s", url)

        try:
            raw = await asyncio.to_thread(_fetch_sync, url, config.timeout_seconds)
            posts = json.loads(raw)
        except Exception as exc:
            return f"Failed to fetch NVIDIA blog posts: {exc}"

        if not isinstance(posts, list) or not posts:
            if query:
                return f"No NVIDIA blog posts found matching '{query}'."
            return "No NVIDIA blog posts returned."

        header = (
            f"NVIDIA Developer Blog posts matching '{query}':"
            if query
            else "Latest NVIDIA Developer Blog posts:"
        )
        lines = [header]
        for i, post in enumerate(posts, 1):
            date = (post.get("date") or "")[:10]  # YYYY-MM-DD
            title = _clean_text(post.get("title", {}).get("rendered", ""))
            link = post.get("link", "")
            excerpt = _clean_text(post.get("excerpt", {}).get("rendered", ""))
            if len(excerpt) > config.excerpt_max_chars:
                excerpt = excerpt[: config.excerpt_max_chars].rstrip() + "..."
            lines.append(f"\n{i}. [{date}] {title}\n   {link}\n   {excerpt}")
        return "\n".join(lines)

    yield FunctionInfo.from_fn(_search, description=_search.__doc__)


Writing router_agent_nb/src/router_agent_nb/nvidia_blog_search_tool.py


<a id="router"></a>
# 4.0) 用 LangGraph 寫 `simple_router_agent`

**這是本份 notebook 的核心。** 上面那兩個工具是熟悉的內容，下面這段才是新的東西：**用 LangGraph 寫一個 NAT workflow function**。


<a id="three-pieces"></a>
## 4.1) LangGraph 三件套：State、Node、Edge

LangGraph 的 API 雖然有十幾個函式，但**核心就 3 個概念**——只要你掌握 State、Node、Edge，剩下都是排列組合。

### 概念 1：State（狀態）

定義你的 workflow 在執行過程中要帶哪些資料。用 Pydantic（或 `TypedDict`）寫一個類別就好：

```python
from pydantic import BaseModel

class RouterState(BaseModel):
    query: str = ""
    intent: str | None = None
    answer: str | None = None
```

這個 state 物件會**自動在節點之間流動**——你不用手動傳參數。

### 概念 2：Node（節點）

一個節點就是一個 **async function**，它的形狀永遠是這樣：

```python
async def my_node(state: RouterState) -> dict:
    # 看一下 state、做點事
    return {"intent": "weather"}  # ← 回傳要更新到 state 的欄位
```

**重點**：節點不直接修改 state，而是 return 一個 dict。LangGraph 會自動把這個 dict **merge 進 state**，再交給下一個節點。

### 概念 3：Edge（邊）

邊有兩種：

| 類型 | API | 用途 |
| --- | --- | --- |
| **普通邊** | `graph.add_edge("A", "B")` | 跑完 A 一定走 B |
| **條件邊** | `graph.add_conditional_edges("A", router_fn, {"x": "B", "y": "C"})` | 跑完 A 之後呼叫 `router_fn(state)`，**它回傳一個字串標籤**，LangGraph 用這個標籤查 dict 決定走哪個下一站 |

最後用 `graph.set_entry_point("first_node")` 指定入口、`graph.compile()` 把整張圖編譯成可執行的物件，就完成了。

接下來我們就用這三個概念，把 2.2 節那張圖蓋出來。


<a id="design-state"></a>
## 4.2) 設計 `RouterState`（狀態）

我們的 router 需要在節點之間傳什麼？看一下 2.2 節的圖：

- `classify_node` 看到 `query`，產出 `intent`
- 條件邊用 `intent` 決定下一站
- `weather_node` / `blog_node` / `meta_node` / `unknown_node` 看到 `query`，產出 `answer`

所以三個欄位就夠了：`query`、`intent`、`answer`。

```python
class RouterState(BaseModel):
    query: str = ""
    intent: str | None = None
    answer: str | None = None
```

**注意**：所有欄位都給預設值，這樣 LangGraph 在啟動時用 `{"query": "..."}` 這種**不完整的 dict** 也能初始化 state——其他欄位先用預設值，等對應節點跑完再被填上。


<a id="design-nodes"></a>
## 4.3) 設計 5 個節點

對照 2.2 節的圖，我們需要寫 5 個節點函式：

```python
async def classify_node(state: RouterState) -> dict:
    # 呼叫 LLM 分類，回傳 {"intent": "weather" / "blog" / "meta" / "unknown"}
    ...

async def weather_node(state: RouterState) -> dict:
    # 呼叫 weather_agent，回傳 {"answer": "..."}
    ...

async def blog_node(state: RouterState) -> dict:
    # 呼叫 blog_agent，回傳 {"answer": "..."}
    ...

async def meta_node(state: RouterState) -> dict:
    return {"answer": "嗨！我是一個小幫手..."}

async def unknown_node(state: RouterState) -> dict:
    return {"answer": "抱歉，我目前只會處理..."}
```

**重點觀察**：每個節點都只回傳「**我這次要更新 state 的哪個欄位**」，不去動其他欄位。LangGraph 會自動 merge。`classify_node` 只寫 `intent`、不碰 `answer`；`weather_node` 只寫 `answer`、不碰 `intent`。


<a id="design-edges"></a>
## 4.4) 設計條件邊：`route_by_intent`

從 `classify_node` 出來之後，要根據 `state.intent` 決定下一站。這就是**條件邊**的工作：

```python
def route_by_intent(state: RouterState) -> str:
    """看一下 state.intent，回傳一個字串標籤。"""
    return state.intent or "unknown"
```

然後在組裝圖的時候用 `add_conditional_edges` 把這個函式接上：

```python
graph.add_conditional_edges(
    "classify",          # 從哪個節點之後做條件判斷
    route_by_intent,     # 用這個函式決定標籤
    {                    # 把標籤對應到下一站節點
        "weather": "weather",
        "blog":    "blog",
        "meta":    "meta",
        "unknown": "unknown",
    },
)
```

**這就是 LangGraph 最常見的 routing 模式**。AI-Q 的 `_build_graph()` 裡有兩條這種條件邊——一條從 intent_classifier 出來分 meta / shallow / deep，一條從 shallow research 出來決定要不要 escalate。


<a id="nat-wrap"></a>
## 4.5) 把所有東西包成 NAT workflow function

現在把三件事 (State, Nodes, Edges) 合起來，再外面包一層 NAT 的 `@register_function`。**請特別注意這個檔案的結構**——上半段是 NAT 的東西、下半段是 LangGraph 的東西、最後用 `_run` 把兩者串起來。


In [6]:
%%writefile router_agent_nb/src/router_agent_nb/simple_router_workflow.py
import logging
from typing import Optional

from langchain_core.messages import HumanMessage
from langchain_core.messages import SystemMessage
from langgraph.graph import END
from langgraph.graph import StateGraph
from pydantic import BaseModel
from pydantic import Field

from nat.builder.builder import Builder
from nat.builder.framework_enum import LLMFrameworkEnum
from nat.builder.function_info import FunctionInfo
from nat.cli.register_workflow import register_function
from nat.data_models.component_ref import FunctionRef
from nat.data_models.component_ref import LLMRef
from nat.data_models.function import FunctionBaseConfig

logger = logging.getLogger(__name__)


# ============================================================
# 1. State（4.2 節）
# ============================================================
class RouterState(BaseModel):
    """State that flows through the LangGraph router."""
    query: str = ""
    intent: Optional[str] = None
    answer: Optional[str] = None


# ============================================================
# 2. NAT config（YAML <-> Python 的橋）
# ============================================================
class SimpleRouterConfig(FunctionBaseConfig, name="simple_router_agent"):
    """A custom workflow that uses LangGraph to route queries to sub-agents."""
    llm_name: LLMRef = Field(description="LLM to use for intent classification.")
    weather_agent: FunctionRef = Field(description="Sub-agent name for weather queries.")
    blog_agent: FunctionRef = Field(description="Sub-agent name for NVIDIA blog queries.")


CLASSIFY_PROMPT = (
    "You are an intent classifier. Look at the user's message and reply with "
    "exactly ONE of these four words, in lowercase, nothing else:\n"
    "  - weather  : about weather, temperature, climate, forecast\n"
    "  - blog     : about NVIDIA developer blog posts / articles / news\n"
    "  - meta     : greeting, asking who you are or what you can do\n"
    "  - unknown  : anything that doesn't fit the categories above\n"
    "Do not explain. Do not add punctuation. One word only."
)


@register_function(config_type=SimpleRouterConfig, framework_wrappers=[LLMFrameworkEnum.LANGCHAIN])
async def simple_router_agent(config: SimpleRouterConfig, builder: Builder):
    """LangGraph-based workflow: classify intent, then dispatch to a sub-agent."""

    # ----- 把 NAT 註冊好的東西拿出來 -----
    llm = await builder.get_llm(config.llm_name, wrapper_type=LLMFrameworkEnum.LANGCHAIN)
    weather_fn = await builder.get_function(config.weather_agent)
    blog_fn = await builder.get_function(config.blog_agent)

    # ============================================================
    # 3. Nodes（4.3 節）
    # ============================================================
    async def classify_node(state: RouterState) -> dict:
        response = await llm.ainvoke(
            [SystemMessage(content=CLASSIFY_PROMPT), HumanMessage(content=state.query)]
        )
        raw = (response.content or "").strip().lower()
        token = raw.split()[0] if raw else "unknown"
        token = "".join(c for c in token if c.isalpha())
        if token not in ("weather", "blog", "meta", "unknown"):
            token = "unknown"
        logger.info("[simple_router] classify_node -> intent=%s", token)
        return {"intent": token}

    async def weather_node(state: RouterState) -> dict:
        result = await weather_fn.ainvoke(state.query)
        answer = result.content if hasattr(result, "content") else str(result)
        return {"answer": answer}

    async def blog_node(state: RouterState) -> dict:
        result = await blog_fn.ainvoke(state.query)
        answer = result.content if hasattr(result, "content") else str(result)
        return {"answer": answer}

    async def meta_node(_state: RouterState) -> dict:
        return {"answer": "嗨！我是一個小幫手，可以幫你查天氣，或搜尋 NVIDIA Developer Blog 的最新文章。"}

    async def unknown_node(_state: RouterState) -> dict:
        return {"answer": "抱歉，我目前只會處理「查天氣」和「查 NVIDIA blog」的問題。"}

    # ============================================================
    # 4. Edge function（4.4 節）
    # ============================================================
    def route_by_intent(state: RouterState) -> str:
        return state.intent or "unknown"

    # ============================================================
    # 5. Assemble the graph
    # ============================================================
    graph = StateGraph(RouterState)

    graph.add_node("classify", classify_node)
    graph.add_node("weather", weather_node)
    graph.add_node("blog", blog_node)
    graph.add_node("meta", meta_node)
    graph.add_node("unknown", unknown_node)

    graph.set_entry_point("classify")

    graph.add_conditional_edges(
        "classify",
        route_by_intent,
        {
            "weather": "weather",
            "blog":    "blog",
            "meta":    "meta",
            "unknown": "unknown",
        },
    )

    for terminal in ("weather", "blog", "meta", "unknown"):
        graph.add_edge(terminal, END)

    compiled_graph = graph.compile()

    # ============================================================
    # 6. NAT 的 _run：把 LangGraph 包成一個普通的 async function
    # ============================================================
    async def _run(query: str) -> str:
        result = await compiled_graph.ainvoke({"query": query})
        return result.get("answer") or "(no answer)"

    yield FunctionInfo.from_fn(
        _run,
        description="Routes user queries to weather or NVIDIA-blog sub-agents using a LangGraph state machine.",
    )


Writing router_agent_nb/src/router_agent_nb/simple_router_workflow.py


讀完上面這個檔案，請花一分鐘確認你**真的看懂了**這幾件事：

1. **NAT 那一層只關心 `_run(query) -> str`**——它不知道也不在乎裡面是 LangGraph、是 if/else、還是擲骰子。
2. **LangGraph 那一層**完全用 State / Node / Edge 三件套組成，沒有任何 NAT 的概念。
3. **連接兩層的是 `_run`**：它把 NAT 的字串輸入餵給 `compiled_graph.ainvoke({"query": query})`，再把結果取出來回傳。

這個「外層 NAT + 內層 LangGraph」的兩層三明治結構，**就是 AI-Q `chat_deepresearcher_agent` 的設計**。你下一份 notebook 看到它的時候，這個結構你已經寫過一次了。


<a id="router-install"></a>
## 4.6) 註冊、安裝、確認

把剛剛三個檔案（兩個工具 + LangGraph workflow）都 import 進 `register.py`，這樣當別人 `pip install` 我們的專案後，NAT CLI 就能在 entry point 掃描時找到它們。


In [7]:
%%writefile -a router_agent_nb/src/router_agent_nb/register.py

from . import weather_lookup_tool
from . import nvidia_blog_search_tool
from . import simple_router_workflow


Appending to router_agent_nb/src/router_agent_nb/register.py


用 `pip install -e` 把整個專案以 editable 模式安裝到當前的 `.venv`。


In [8]:
!pip install -e router_agent_nb

Obtaining file:///localhome/local-clchiu/NeMo-Agent-Toolkit/examples/notebooks/zh-TW/router_agent_nb
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for router_agent_nb (pyproject.toml) ... done
  Created wheel for router_agent_nb: filename=router_agent_nb-0.1.0-0.editable-py3-none-any.whl size=1698 sha256=098748a7049584703db0ab1f0d3ebc0b84526cb548130033e7ba63f3ef10afd1
  Stored in directory: /tmp/pip-ephem-wheel-cache-h_3ok99_/wheels/da/71/4d/8abdd0ab7812df1f3b9ae1970d2bde62ca5ccfc7c136e61e5f
Successfully built router_agent_nb
  Attempting uninstall: router_agent_nb
    Found existing installation: router_agent_nb 0.1.0
    Uninstalling router_agent_nb-0.1.0:
      Successfully uninstalled router_agent_nb-0.1.0


安裝完成後，我們用 `nat info components` 確認 NAT 看得到我們剛剛註冊的東西：


In [9]:
!nat info components -t function -q simple_router_agent

                               NAT Search Results                               
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃ package        ┃ version ┃ component_type ┃ component_name  ┃ description    ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│ router_agent_n │ 0.1.0   │ function       │ simple_router_a │ A custom       │
│ b              │         │                │ gent            │ workflow that  │
│                │         │                │                 │ uses LangGraph │
│                │         │                │                 │ to route       │
│                │         │                │                 │ queries to     │
│                │         │                │                 │ sub-agents.    │
│                │         │                │                 │                │
│                │         │                │                 │   Args:        │
│                │         │

你應該會看到 `simple_router_agent` 出現在輸出裡，並有我們在 docstring 寫的描述。同樣可以查 `weather_lookup` 與 `nvidia_blog_search`：


In [10]:
!nat info components -t function -q weather_lookup
!nat info components -t function -q nvidia_blog_search

                               NAT Search Results                               
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃ package         ┃ version ┃ component_type ┃ component_name ┃ description    ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│ router_agent_nb │ 0.1.0   │ function       │ weather_lookup │ Look up the    │
│                 │         │                │                │ current        │
│                 │         │                │                │ weather for a  │
│                 │         │                │                │ city from      │
│                 │         │                │                │ wttr.in (no    │
│                 │         │                │                │ API key        │
│                 │         │                │                │ needed).       │
│                 │         │                │                │                │
│                 │         

<a id="yaml"></a>
# 5.0) 把所有東西串成一份 YAML

到目前為止我們**只寫了 Python**。現在用 YAML 來把這些零件「組裝」成一個完整 workflow。


<a id="yaml-before"></a>
## 5.1) Before：所有工具塞給單一 agent

如果不做意圖分類，直接把兩個工具丟給**單一**內建 agent，YAML 會長這樣（**這份只是給你對照用，我們不會真的執行它**）：

```yaml
llms:
  nim_llm:
    _type: nim
    model_name: nvidia/nemotron-3-super-120b-a12b
    temperature: 0.0
    api_key: $NVIDIA_API_KEY

functions:
  weather_lookup:
    _type: weather_lookup
  nvidia_blog_search:
    _type: nvidia_blog_search

workflow:
  _type: tool_calling_agent      # ← NAT 內建（也可以寫 react_agent）
  tool_names: [weather_lookup, nvidia_blog_search]
  llm_name: nim_llm
```

問題我們在 2.1 節說過了：query 都丟給同一顆 agent，meta / unknown query 也會跑工具判斷流程。


<a id="yaml-after"></a>
## 5.2) After：兩個 sub-agent + 一個 LangGraph router

下面這份才是我們真的要跑的。請特別注意：

1. `functions:` 區塊多了 `weather_agent` 跟 `blog_agent`——它們的 `_type` 是 **`tool_calling_agent`**（NAT 內建的兩種 agent 之一，**跟 `react_agent` 形狀完全一樣**）。它們被當成「sub-agent」放在 `functions:` 區塊。
2. `workflow:` 區塊的 `_type` 從 `tool_calling_agent` 換成了 **`simple_router_agent`**——我們剛剛寫的 LangGraph 版本。

> **💡 為什麼這份 notebook 用 `tool_calling_agent` 而前一份 notebook 用 `react_agent`？**
>
> 這兩個是 NAT 內建的兩種 agent type，**功能相同、外部 YAML 介面完全一樣**（都吃 `tool_names`、`llm_name`、`description`），差別在它們**怎麼跟 LLM 互動**：
>
> | | `react_agent` | `tool_calling_agent` |
> | --- | --- | --- |
> | 與 LLM 互動方式 | **文字 prompt**：要 LLM 寫 `Thought: ... Action: ...` 再用正則 parse | **API 原生**：直接用 OpenAI / NIM 的 `tools=[...]` 參數 |
> | 適合的 LLM | `meta/llama-3.3-70b-instruct`、Claude、有 ReAct 訓練的模型 | `nvidia/nemotron-3-super-120b-a12b`、GPT-4、大部分新 NIM |
> | log 看起來像 | `Thought: ...` / `Action: ...` 文字 | 結構化 tool call 物件 |
>
> 這份 notebook 搭配的是 **Nemotron-3 Super 120B-A12B**——它是設計給原生 function calling 用的，跑 ReAct 文字 prompt 不擅長。所以我們用 `tool_calling_agent`。如果你想換成 llama-3.3-70b-instruct，**只要把 `_type` 改回 `react_agent` 即可，其他完全不變**。


In [11]:
%%writefile router_agent_nb/configs/config.yml
llms:
  nim_llm:
    _type: nim
    model_name: nvidia/nemotron-3-super-120b-a12b
    temperature: 0.0
    max_tokens: 2048
    api_key: $NVIDIA_API_KEY

functions:
  # ---- 原子工具 ----
  weather_lookup:
    _type: weather_lookup
  nvidia_blog_search:
    _type: nvidia_blog_search

  # ---- 兩個專業 sub-agent（每個都是「方式 A」風格的內建 agent）----
  weather_agent:
    _type: tool_calling_agent
    tool_names: [weather_lookup]
    llm_name: nim_llm
    description: "Handles weather queries."
    verbose: true

  blog_agent:
    _type: tool_calling_agent
    tool_names: [nvidia_blog_search]
    llm_name: nim_llm
    description: "Handles NVIDIA Developer Blog queries."
    verbose: true

# ---- 我們自己寫的 LangGraph 編排器（「方式 B」）----
workflow:
  _type: simple_router_agent
  llm_name: nim_llm
  weather_agent: weather_agent   # 對應 functions: 區塊的名字
  blog_agent: blog_agent


Overwriting router_agent_nb/configs/config.yml


**為什麼這份 YAML 很重要？**

請特別停下來看一下 `workflow:` 區塊：

```yaml
workflow:
  _type: simple_router_agent
  llm_name: nim_llm
  weather_agent: weather_agent
  blog_agent: blog_agent
```

跟 4.5 節我們寫的 Python `SimpleRouterConfig` 對照：

```python
class SimpleRouterConfig(FunctionBaseConfig, name="simple_router_agent"):
    llm_name: LLMRef = Field(...)
    weather_agent: FunctionRef = Field(...)
    blog_agent: FunctionRef = Field(...)
```

每一個 YAML 欄位都對應到 Pydantic config 上的一個 field。**NAT 用 Pydantic 自動把 YAML 解析成這個 config 物件**——這就是為什麼前一份 notebook 寫工具時、跟這份 notebook 寫 workflow 時，「形狀」這麼一致：因為它們**真的是同一種東西**。


<a id="run"></a>
# 6.0) 跑跑看

下面四個 cell 會丟四種不同類型的 query 給 router。你可以在 log 裡找到 `[simple_router] classify_node -> intent=...` 這一行，確認 LangGraph 真的把意圖分對了、條件邊真的把它路由到對應的節點。


<a id="run-meta"></a>
## 6.1) Meta query：`classify → meta → END`

LangGraph 跑的路徑：

```
START → classify_node (intent=meta) → meta_node → END
```

預期 log：
```
[simple_router] classify_node -> intent=meta
```
然後回覆我們在 4.5 節寫的那句問候，**完全不會跑 weather_agent 或 blog_agent**。


In [12]:
!nat run --config_file router_agent_nb/configs/config.yml --input "你好"

2026-05-18 14:11:41 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'router_agent_nb/configs/config.yml'

Configuration Summary:
--------------------
Workflow Type: simple_router_agent
Number of Functions: 4
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2026-05-18 14:11:41 - INFO     - nat.runtime.session:329 - Shared workflow built (entry_function=None)
2026-05-18 14:11:51 - INFO     - router_agent_nb.simple_router_workflow:74 - [simple_router] classify_node -> intent=meta
2026-05-18 14:11:51 - INFO     - nat.front_ends.console.console_front_end_plugin:162 - --------------------------------------------------
Workflow Result:
嗨!我是一個小幫手,可以幫你查天氣,或搜尋 NVIDIA Developer Blog 的最新文章。
--------------------------------------------------


<a id="run-weather"></a>
## 6.2) Weather query：`classify → weather → END`

LangGraph 跑的路徑：

```
START → classify_node (intent=weather) → weather_node
        → weather_agent (tool_calling_agent) → weather_lookup(city="Taipei") → END
```


In [13]:
!nat run --config_file router_agent_nb/configs/config.yml --input "請問今天台北市的天氣如何?"

2026-05-18 14:11:53 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'router_agent_nb/configs/config.yml'

Configuration Summary:
--------------------
Workflow Type: simple_router_agent
Number of Functions: 4
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2026-05-18 14:11:53 - INFO     - nat.runtime.session:329 - Shared workflow built (entry_function=None)
2026-05-18 14:11:54 - INFO     - router_agent_nb.simple_router_workflow:74 - [simple_router] classify_node -> intent=weather
2026-05-18 14:11:59 - INFO     - nat.plugins.langchain.agent.tool_calling_agent.agent:173 - 
------------------------------
[AGENT]
Agent input: 請問今天台北市的天氣如何?
Agent's thoughts: 
content='' additional_kwargs={'reasoning_content': 'The user asks in Chinese: "請問今天台北市的天氣如何?" meaning "What is the weather like today in Taipei?" We need

<a id="run-blog"></a>
## 6.3) Blog query：`classify → blog → END`

LangGraph 跑的路徑：

```
START → classify_node (intent=blog) → blog_node
        → blog_agent (tool_calling_agent) → nvidia_blog_search(query="agentic AI") → END
```


In [14]:
!nat run --config_file router_agent_nb/configs/config.yml --input "請幫我找最新關於 agentic AI 的 NVIDIA Tech Blog 文章"

2026-05-18 14:12:09 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'router_agent_nb/configs/config.yml'

Configuration Summary:
--------------------
Workflow Type: simple_router_agent
Number of Functions: 4
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2026-05-18 14:12:09 - INFO     - nat.runtime.session:329 - Shared workflow built (entry_function=None)
2026-05-18 14:12:11 - INFO     - router_agent_nb.simple_router_workflow:74 - [simple_router] classify_node -> intent=blog
2026-05-18 14:12:13 - INFO     - nat.plugins.langchain.agent.tool_calling_agent.agent:173 - 
------------------------------
[AGENT]
Agent input: 請幫我找最新關於 agentic AI 的 NVIDIA Tech Blog 文章
Agent's thoughts: 
content='' additional_kwargs={'reasoning_content': 'We need to search NVIDIA Developer Blog for latest about agentic AI. Use the

<a id="run-unknown"></a>
## 6.4) Unknown query：`classify → unknown → END`

我們的 router 設計上**只處理天氣與日期**。其他類型的問題不會浪費 LLM 與工具，由 `unknown_node` 直接回拒。

LangGraph 跑的路徑：

```
START → classify_node (intent=unknown) → unknown_node → END
```


In [15]:
!nat run --config_file router_agent_nb/configs/config.yml --input "幫我寫一首關於秋天的詩"

2026-05-18 14:12:38 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'router_agent_nb/configs/config.yml'

Configuration Summary:
--------------------
Workflow Type: simple_router_agent
Number of Functions: 4
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2026-05-18 14:12:39 - INFO     - nat.runtime.session:329 - Shared workflow built (entry_function=None)
2026-05-18 14:12:40 - INFO     - router_agent_nb.simple_router_workflow:74 - [simple_router] classify_node -> intent=unknown
2026-05-18 14:12:40 - INFO     - nat.front_ends.console.console_front_end_plugin:162 - --------------------------------------------------
Workflow Result:
抱歉,我目前只會處理「查天氣」和「查 NVIDIA blog」的問題。
--------------------------------------------------


<a id="bridge-to-aiq"></a>
# 7.0) 總結

恭喜你寫完了一個用 LangGraph 寫的 NAT workflow！花一分鐘整理一下這份 notebook 你帶走了什麼：

| 你現在能寫的東西 | 在哪一節學到 |
| --- | --- |
| 在 NAT 裡寫一個自訂工具（`@register_function` + 呼叫外部 HTTP API） | 3.2 / 3.3 |
| 設計一個 LangGraph 狀態：什麼進、什麼出，欄位 schema 怎麼定 | 4.2 |
| 把節點寫成 async function、`return` 一個 dict 去更新 state | 4.3 |
| 用條件邊（`add_conditional_edges`）＋ mapping dict 做 intent routing | 4.4 |
| 在 workflow function 裡用 `builder.get_function(...)` 重用 sub-agent | 4.5 |
| 把整個 LangGraph workflow 包成一個 NAT function、再用 YAML 把所有東西串起來 | 4.5 / 5.2 |
| 知道什麼時候該用 `tool_calling_agent`、什麼時候用 `react_agent` | 5.2 提示框 |